# 07 — Multi-palé first-fit

No entrena. Con dos palés, puntos extremos reparte la carga. Este paso
envuelve la política (heurístico o el PPO del `05`) en first-fit:
mientras el palé abierto admita algo, el vacío no es candidata.

Diseño emparejado `libre` / `consolida`. Siguiente: `08_producto_execute.ipynb`.


## Qué problema se está atacando

La Fase 1 puso dos pallets idénticos a disposición de cada motor y descubrió un defecto: en lugar de
llenar el primero y usar el segundo como desborde, los motores **reparten la carga casi por mitad**.
El caso más claro es el pedido `00100408`, que la Fase 0 ya había demostrado que cabe entero en un
solo pallet: con dos disponibles, el heurístico y el MLP lo parten 14/12 y el cubo se hunde del
64.6 % al 32.3 %.

## Qué hace este notebook

Mide cuánto de esa brecha se cierra **sin reentrenar y sin tocar el modelo**. Se envuelve la política
existente en `ConsolidatingPolicy`, que impone disciplina *first-fit*: mientras el pallet abierto
admita algo, las candidatas del siguiente ni se ponen sobre la mesa. La política interna (greedy o el
PPO de producción / cinta) sigue eligiendo la pose exactamente como antes.

El diseño es **emparejado**: cada brazo corre dos veces, `libre` y `consolida`, con el mismo
checkpoint, el mismo encoder v1, el mismo bucle, el mismo validador y las mismas métricas. La única
diferencia entre los dos miembros de cada par es la disciplina de contenedor.

## Qué vas a decidir al final

El Paso A no es un arreglo definitivo: es la medición que dice **dónde estaba el problema**.

- Si la disciplina recupera los pallets y el cubo, la brecha era del **espacio de acciones**,
  no de la política: el producto con dos pallets puede usar la envolvente.
- Si la disciplina **no** mueve las cifras, la culpa es de la política de dos contenedores.
  Eso no reabre el PPO: el fine-tuning sobre el encoder v1 ya está cerrado.

La última celda sella el resultado en el registro de versiones, de modo que este número quede atado
al commit y al notebook que lo produjo.

In [1]:
from pathlib import Path
import sys

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

HERE = Path.cwd().resolve()
if HERE.name == "notebooks":
    ML_ROOT = HERE.parent
else:
    ML_ROOT = HERE if (HERE / "src_ml").is_dir() else Path("/home/edmundo/packing-services/online_policy_ml")

REPO_ROOT = ML_ROOT.parent
for p in (ML_ROOT / "src_ml", REPO_ROOT / "src"):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from bootstrap import setup
setup()

import json
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

print("ML_ROOT   =", ML_ROOT)
print("REPO_ROOT =", REPO_ROOT)

ML_ROOT   = /home/edmundo/packing-services/online_policy_ml
REPO_ROOT = /home/edmundo/packing-services


## Comprobaciones de entrada

Antes de medir, el notebook verifica que no se está tocando nada de lo que ya funciona:

- El encoder sigue siendo **v1 con 35 features**. Si cambiara, los checkpoints entrenados dejarían de
  ser válidos y la comparación no diría nada.
- Los checkpoints de producción (`mlp_v1_p1s1_ppo.pt` y `mlp_v1_p3s2.pt`) existen y se anota su
  fecha de modificación. La última celda vuelve a comprobarla: si cambiara, el flujo habría
  reentrenado algo y eso sería un error.
- Los cinco pedidos son el **holdout de producto**, que nunca entró en entrenamiento ni en ajuste.

In [2]:
from consolidation import PASO_A_ARMS, pair_deltas, run_arm, run_paso_a
from paths import HOLDOUT_DIR, REPORTS_DIR
from registry import (
    STATUS_EN_CURSO,
    STATUS_VALIDADO,
    load_sealed_report,
    registry_table,
    seal_run,
)
from splits import load_orders

from packing_services.online.features import FEATURE_DIM, FEATURE_VERSION
from packing_services.online.learned.production import (
    CINTA_MODEL_PATH,
    DEFAULT_MODEL_PATH,
    PRODUCT_HOLDOUT_ORDER_IDS,
)

FLOW_ID = "eval_pasoA_consolidacion"
NOTEBOOK = "online_policy_ml/notebooks/07_multipallet_firstfit.ipynb"
CONTAINERS = 2

assert FEATURE_VERSION == 1, "este flujo no cambia el encoder"
assert FEATURE_DIM == 35, "este flujo no cambia el encoder"

MODELS = {name: REPO_ROOT / name for name in (DEFAULT_MODEL_PATH, CINTA_MODEL_PATH)}
for name, path in MODELS.items():
    assert path.is_file(), f"falta el checkpoint de produccion {name}"
MTIMES_ANTES = {name: path.stat().st_mtime for name, path in MODELS.items()}

ORDERS = load_orders(HOLDOUT_DIR / "5_bed-bpp.json")
ORDER_IDS = sorted(ORDERS)
assert set(ORDER_IDS) == set(PRODUCT_HOLDOUT_ORDER_IDS), "el holdout no coincide"

print("encoder v1 con", FEATURE_DIM, "features")
print("produccion p1s1:", DEFAULT_MODEL_PATH)
print("cinta p3s2     :", CINTA_MODEL_PATH)
print("contenedores disponibles por pedido:", CONTAINERS)
print("brazos:", len(PASO_A_ARMS), "(", len(PASO_A_ARMS) // 2, "pares libre/consolida )")
for arm in PASO_A_ARMS:
    print(f"  {arm.label:<24} p={arm.lookahead_p} s={arm.select_s} disciplina={arm.discipline}")
print()
print("pedidos de holdout:")
for oid in ORDER_IDS:
    props = ORDERS[oid].get("properties", {})
    print(f"  {oid}  {len(ORDERS[oid]['item_sequence']):>3} items  target={props.get('target')}")

encoder v1 con 35 features
produccion p1s1: online_policy_ml/artifacts/models/mlp_v1_p1s1_ppo.pt
cinta p3s2     : online_policy_ml/artifacts/models/mlp_v1_p3s2.pt
contenedores disponibles por pedido: 2
brazos: 6 ( 3 pares libre/consolida )
  greedy p1s1 libre        p=1 s=1 disciplina=libre
  greedy p1s1 consolida    p=1 s=1 disciplina=consolida
  rl_p1s1 libre            p=1 s=1 disciplina=libre
  rl_p1s1 consolida        p=1 s=1 disciplina=consolida
  mlp_p3s2 libre           p=3 s=2 disciplina=libre
  mlp_p3s2 consolida       p=3 s=2 disciplina=consolida

pedidos de holdout:
  00100001   44 items  target=rollcontainer
  00100002   38 items  target=rollcontainer
  00100003   34 items  target=rollcontainer
  00100004   58 items  target=euro-pallet
  00100408   26 items  target=euro-pallet


## Por qué el modelo reparte la carga

Esto no es un bug de código: es una consecuencia de cómo se entrenó. El encoder v1 expone tres
señales relevantes aquí:

| Feature | Qué mide |
|---|---|
| `n_packed_n` | ítems ya colocados en **toda** la sesión |
| `used_height_n` | altura usada **del contenedor de esa candidata** |
| `loaded_weight_n` | peso cargado **del contenedor de esa candidata** |

La conjunción "este pallet está vacío **pero** ya llevo 14 ítems colocados" es perfectamente
representable con esas tres entradas, y el MLP de 35→64→ReLU→1 tiene capacidad de sobra para
aprenderla. El problema es que esa conjunción **nunca ocurrió en entrenamiento**: todas las
instancias P2O tenían un solo contenedor, así que `used_height_n = 0` coincidía siempre con
`n_packed_n = 0`, es decir con el primer ítem del episodio, que es una colocación buena.

El modelo aprendió "contenedor vacío → adelante". Con dos pallets esa misma firma aparece en cada
paso para el pallet 2, y el modelo la lee como "empiezo un episodio, coloca libremente".

A esto se suma que la puntuación de puntos extremos premia el área de contacto, y el suelo vacío del
segundo pallet ofrece contacto igual a la huella completa de la caja con `z = 0`. Por eso también el
greedy, que no usa el modelo, comete el mismo error.

La celda siguiente reproduce el defecto en un solo pedido para que lo veas antes de medir todo.

In [3]:
DEMO_ID = "00100408"

demo = [
    run_arm(ORDERS, DEMO_ID, arm, containers=CONTAINERS)
    for arm in PASO_A_ARMS
    if arm.family == "greedy"
]
demo_df = pd.DataFrame(demo)[
    ["arm", "is_valid", "items_packed", "items_unpacked", "containers_used",
     "volume_utilization", "load_height_mm", "spread", "seconds"]
]
print(f"Pedido {DEMO_ID}: la Fase 0 demostro que cabe entero en UN pallet al 64.6%\n")
display(demo_df)

libre = demo[0]
cons = demo[1]
print(f"libre     -> {libre['containers_used']} pallet(s), reparto {libre['spread']}")
print(f"consolida -> {cons['containers_used']} pallet(s), reparto {cons['spread']}")
print(f"cubo: {libre['volume_utilization']:.1%} -> {cons['volume_utilization']:.1%}")

Pedido 00100408: la Fase 0 demostro que cabe entero en UN pallet al 64.6%



,arm,is_valid,items_packed,items_unpacked,containers_used,volume_utilization,load_height_mm,spread,seconds
0,greedy p1s1 libre,True,26,0,2,0.3232,"2,000.0000","{'EURO_PALLET-1': 14, 'EURO_PALLET-2': 12}",0.8349
1,greedy p1s1 consolida,True,26,0,1,0.6464,"2,000.0000",{'EURO_PALLET-1': 26},0.7088


libre     -> 2 pallet(s), reparto {'EURO_PALLET-1': 14, 'EURO_PALLET-2': 12}
consolida -> 1 pallet(s), reparto {'EURO_PALLET-1': 26}
cubo: 32.3% -> 64.6%


## El experimento

Seis brazos sobre los cinco pedidos del holdout, con dos pallets disponibles: 30 corridas.

| Motor | Presupuesto | Checkpoint |
|---|---|---|
| greedy | p=1, s=1 | ninguno |
| RL de producción | p=1, s=1 | `mlp_v1_p1s1_ppo.pt` |
| MLP de cinta | p=3, s=2 | `mlp_v1_p3s2.pt` |

Cada uno corre `libre` y `consolida`. El greedy con lookahead `p=3` queda fuera a propósito: en la
Fase 0 tardó 95 s en un pedido de 26 ítems y más de 22 minutos en uno de 44, así que no es operable.

La celda tarda unos minutos e imprime el progreso pedido a pedido. Todo pasa por
`run_online_loop` para colocar y por `build_solution` para validar y calcular métricas, que es el
mismo camino del execute de producción.

In [4]:
rows = run_paso_a(ORDERS, ORDER_IDS, containers=CONTAINERS, verbose=True)
print()
print(len(rows), "corridas =", len(ORDER_IDS), "pedidos x", len(PASO_A_ARMS), "brazos")

df = pd.DataFrame(rows)
assert df["is_valid"].all(), "hay soluciones invalidas: revisar antes de seguir"
assert (df["constraint_violations"] == 0).all(), "hay violaciones de restricciones"
print("todas las soluciones son validas y sin violaciones")

00100001 · greedy p1s1 libre      pallets=2 emp=44/44 util=39.3% t=3.03s
00100001 · greedy p1s1 consolida  pallets=2 emp=44/44 util=39.3% t=1.43s
00100001 · rl_p1s1 libre          pallets=2 emp=44/44 util=39.3% t=1.51s
00100001 · rl_p1s1 consolida      pallets=2 emp=44/44 util=39.3% t=1.15s
00100001 · mlp_p3s2 libre         pallets=2 emp=44/44 util=39.3% t=4.77s
00100001 · mlp_p3s2 consolida     pallets=2 emp=44/44 util=39.3% t=2.29s
00100002 · greedy p1s1 libre      pallets=2 emp=38/38 util=40.1% t=1.67s
00100002 · greedy p1s1 consolida  pallets=2 emp=38/38 util=40.1% t=0.93s
00100002 · rl_p1s1 libre          pallets=2 emp=38/38 util=40.1% t=1.56s
00100002 · rl_p1s1 consolida      pallets=2 emp=38/38 util=40.1% t=1.00s
00100002 · mlp_p3s2 libre         pallets=2 emp=38/38 util=40.1% t=2.82s
00100002 · mlp_p3s2 consolida     pallets=2 emp=38/38 util=40.1% t=1.70s
00100003 · greedy p1s1 libre      pallets=2 emp=34/34 util=43.9% t=1.12s
00100003 · greedy p1s1 consolida  pallets=2 emp=34/

## Tabla por pedido

Cómo leer las columnas:

- `containers_used`: pallets que el motor decidió abrir. **Menos es mejor**, siempre que el pedido
  salga completo.
- `volume_utilization`: cubo sobre los contenedores **efectivamente usados**. Abrir un pallet de más
  lo parte por la mitad, así que esta columna ya castiga el reparto.
- `load_height_mm` y `envelope_utilization`: altura de carga alcanzada y cubo dentro de esa altura.
  Sirven cuando el pedido entero cabe y el cubo empata entre motores.

In [5]:
COLS = [
    "arm", "is_valid", "items_packed", "items_unpacked", "containers_used",
    "volume_utilization", "load_height_mm", "envelope_utilization", "seconds",
]

for oid in ORDER_IDS:
    sub = df[df["order_id"] == oid]
    n_items = int(sub["n_items"].iloc[0])
    print(f"--- {oid} · {n_items} items ---")
    display(
        sub[COLS]
        .sort_values(["containers_used", "volume_utilization"], ascending=[True, False])
        .reset_index(drop=True)
    )

--- 00100001 · 44 items ---


,arm,is_valid,items_packed,items_unpacked,containers_used,volume_utilization,load_height_mm,envelope_utilization,seconds
0,greedy p1s1 libre,True,44,0,2,0.3925,"2,000.0000",0.3925,3.0305
1,greedy p1s1 consolida,True,44,0,2,0.3925,"2,000.0000",0.4715,1.4287
2,rl_p1s1 libre,True,44,0,2,0.3925,"2,000.0000",0.4423,1.5082
3,rl_p1s1 consolida,True,44,0,2,0.3925,"2,000.0000",0.4787,1.1469
4,mlp_p3s2 libre,True,44,0,2,0.3925,"1,970.0000",0.4026,4.7722
5,mlp_p3s2 consolida,True,44,0,2,0.3925,"1,970.0000",0.6256,2.2944


--- 00100002 · 38 items ---


,arm,is_valid,items_packed,items_unpacked,containers_used,volume_utilization,load_height_mm,envelope_utilization,seconds
0,greedy p1s1 libre,True,38,0,2,0.4012,"2,000.0000",0.4032,1.6705
1,greedy p1s1 consolida,True,38,0,2,0.4012,"2,000.0000",0.5900,0.9294
2,rl_p1s1 libre,True,38,0,2,0.4012,"2,000.0000",0.4084,1.5614
3,rl_p1s1 consolida,True,38,0,2,0.4012,"2,000.0000",0.4169,1.0038
4,mlp_p3s2 libre,True,38,0,2,0.4012,"2,000.0000",0.4032,2.8209
5,mlp_p3s2 consolida,True,38,0,2,0.4012,"1,980.0000",0.4053,1.6957


--- 00100003 · 34 items ---


,arm,is_valid,items_packed,items_unpacked,containers_used,volume_utilization,load_height_mm,envelope_utilization,seconds
0,greedy p1s1 libre,True,34,0,2,0.4387,"1,940.0000",0.4546,1.1220
1,greedy p1s1 consolida,True,34,0,2,0.4387,"2,000.0000",0.6200,0.4861
2,rl_p1s1 libre,True,34,0,2,0.4387,"2,000.0000",0.6072,0.7269
3,rl_p1s1 consolida,True,34,0,2,0.4387,"2,000.0000",0.6523,0.6446
4,mlp_p3s2 libre,True,34,0,2,0.4387,"2,000.0000",0.4387,1.4762
5,mlp_p3s2 consolida,True,34,0,2,0.4387,"2,000.0000",0.5028,1.2217


--- 00100004 · 58 items ---


,arm,is_valid,items_packed,items_unpacked,containers_used,volume_utilization,load_height_mm,envelope_utilization,seconds
0,greedy p1s1 consolida,True,58,0,1,0.6142,"2,000.0000",0.6142,6.8214
1,rl_p1s1 consolida,True,58,0,1,0.6142,"1,970.0000",0.6235,6.4556
2,mlp_p3s2 consolida,True,58,0,1,0.6142,"2,000.0000",0.6142,14.7681
3,greedy p1s1 libre,True,58,0,2,0.3071,"2,000.0000",0.3071,9.9550
4,rl_p1s1 libre,True,58,0,2,0.3071,"1,970.0000",0.5388,6.8424
5,mlp_p3s2 libre,True,58,0,2,0.3071,"2,000.0000",0.3441,16.6999


--- 00100408 · 26 items ---


,arm,is_valid,items_packed,items_unpacked,containers_used,volume_utilization,load_height_mm,envelope_utilization,seconds
0,greedy p1s1 consolida,True,26,0,1,0.6464,"2,000.0000",0.6464,0.7136
1,rl_p1s1 consolida,True,26,0,1,0.6464,"1,970.0000",0.6562,0.5215
2,mlp_p3s2 consolida,True,26,0,1,0.6464,"1,975.0000",0.6546,1.1533
3,greedy p1s1 libre,True,26,0,2,0.3232,"2,000.0000",0.3232,0.8237
4,rl_p1s1 libre,True,26,0,2,0.3232,"1,970.0000",0.5455,0.5979
5,mlp_p3s2 libre,True,26,0,2,0.3232,"1,970.0000",0.3358,1.7025


## La lectura que decide: cada brazo contra su gemelo

Aquí está la respuesta del Paso A. Cada renglón compara el **mismo motor con el mismo checkpoint y el
mismo presupuesto** consigo mismo, cambiando solo la disciplina de contenedor.

- `pallets_delta` negativo significa que la disciplina **ahorró** pallets.
- `util_delta` positivo significa que el cubo **mejoró**.
- `fuera_delta` positivo sería una regresión: la disciplina habría dejado más carga en el muelle.
  Esa es la única forma en que el Paso A podría hacer daño, y hay que mirarla.

In [6]:
deltas = pd.DataFrame(pair_deltas(rows))
display(deltas)

resumen = (
    deltas.groupby("motor")
    .agg(
        pedidos=("order_id", "count"),
        pallets_libre=("pallets_libre", "sum"),
        pallets_consolida=("pallets_consolida", "sum"),
        fuera_libre=("fuera_libre", "sum"),
        fuera_consolida=("fuera_consolida", "sum"),
        util_libre=("util_libre", "mean"),
        util_consolida=("util_consolida", "mean"),
        seg_libre=("seg_libre", "mean"),
        seg_consolida=("seg_consolida", "mean"),
    )
    .reset_index()
)
resumen["pallets_ahorrados"] = resumen["pallets_libre"] - resumen["pallets_consolida"]
resumen["util_delta"] = resumen["util_consolida"] - resumen["util_libre"]

print("\nResumen por motor (totales de pallets e items, medias de cubo y tiempo):")
display(
    resumen[
        ["motor", "pedidos", "pallets_libre", "pallets_consolida", "pallets_ahorrados",
         "fuera_libre", "fuera_consolida", "util_libre", "util_consolida", "util_delta",
         "seg_libre", "seg_consolida"]
    ]
)

,order_id,motor,n_items,pallets_libre,pallets_consolida,pallets_delta,util_libre,util_consolida,util_delta,fuera_libre,fuera_consolida,fuera_delta,valido_libre,valido_consolida,seg_libre,seg_consolida
0,00100001,aprendida#p1s1,44,2,2,0,0.3925,0.3925,0.0000,0,0,0,True,True,1.5082,1.1469
1,00100002,aprendida#p1s1,38,2,2,0,0.4012,0.4012,0.0000,0,0,0,True,True,1.5614,1.0038
2,00100003,aprendida#p1s1,34,2,2,0,0.4387,0.4387,0.0000,0,0,0,True,True,0.7269,0.6446
3,00100004,aprendida#p1s1,58,2,1,-1,0.3071,0.6142,0.3071,0,0,0,True,True,6.8424,6.4556
4,00100408,aprendida#p1s1,26,2,1,-1,0.3232,0.6464,0.3232,0,0,0,True,True,0.5979,0.5215
5,00100001,aprendida#p3s2,44,2,2,0,0.3925,0.3925,0.0000,0,0,0,True,True,4.7722,2.2944
6,00100002,aprendida#p3s2,38,2,2,0,0.4012,0.4012,0.0000,0,0,0,True,True,2.8209,1.6957
7,00100003,aprendida#p3s2,34,2,2,0,0.4387,0.4387,0.0000,0,0,0,True,True,1.4762,1.2217
8,00100004,aprendida#p3s2,58,2,1,-1,0.3071,0.6142,0.3071,0,0,0,True,True,16.6999,14.7681
9,00100408,aprendida#p3s2,26,2,1,-1,0.3232,0.6464,0.3232,0,0,0,True,True,1.7025,1.1533



Resumen por motor (totales de pallets e items, medias de cubo y tiempo):


,motor,pedidos,pallets_libre,pallets_consolida,pallets_ahorrados,fuera_libre,fuera_consolida,util_libre,util_consolida,util_delta,seg_libre,seg_consolida
0,aprendida#p1s1,5,10,8,2,0,0,0.3725,0.4986,0.1261,2.2474,1.9545
1,aprendida#p3s2,5,10,8,2,0,0,0.3725,0.4986,0.1261,5.4943,4.2266
2,greedy#p1s1,5,10,8,2,0,0,0.3725,0.4986,0.1261,3.3203,2.0759


## Contra la Fase 1 ya sellada

El brazo `libre` del **greedy** y de la **cinta** (`mlp_v1_p3s2.pt`) debe reproducir la Fase 1:
mismo código, mismos checkpoints. El aprendido p=1 s=1 **no**: la Fase 1 usó imitación
(`mlp_v1_p1s1.pt`) y hoy el producto carga PPO (`mlp_v1_p1s1_ppo.pt`). Ese desvío se anota;
no invalida la tabla.

La celda busca primero el resultado en el registro de versiones y, si aún no está sellado, cae al
reporte suelto `artifacts/reports/08_holdout_multipallet.json`.

In [7]:
try:
    fase1 = load_sealed_report("eval_fase1_multipallet")
    origen = "registro sellado (eval_fase1_multipallet)"
except FileNotFoundError:
    fase1 = json.loads((REPORTS_DIR / "08_holdout_multipallet.json").read_text(encoding="utf-8"))
    origen = "reporte suelto 08_holdout_multipallet.json"

print("baseline tomado de:", origen)

# La Fase 1 etiqueta los motores online como online:PALLETIZATION/<algoritmo>#<variante>.
# p=1 s=1 aprendido era imitación; hoy es PPO. Solo greedy y cinta deben coincidir.
EQUIVALENCIAS = {
    "greedy#p1s1": "online:PALLETIZATION/online_3d_bpp_heuristic#p1s1",
    "aprendida#p1s1": "online:PALLETIZATION/drl_policy_3d_bpp#mlp_p1s1",
    "aprendida#p3s2": "online:PALLETIZATION/drl_policy_3d_bpp#mlp_p3s2",
}
DEBE_REPRODUCIR = {"greedy#p1s1", "aprendida#p3s2"}
por_motor_fase1 = {row["engine"]: row for row in fase1["summary"]["engines"]}

check = []
for motor, etiqueta in EQUIVALENCIAS.items():
    previo = por_motor_fase1.get(etiqueta)
    actual = resumen[resumen["motor"] == motor]
    if previo is None or actual.empty:
        continue
    fila = actual.iloc[0]
    check.append(
        {
            "motor": motor,
            "pallets_fase1": previo["containers_used"],
            "pallets_libre_hoy": int(fila["pallets_libre"]),
            "util_fase1": previo["mean_volume_utilization"],
            "util_libre_hoy": float(fila["util_libre"]),
            "pallets_consolida_hoy": int(fila["pallets_consolida"]),
            "util_consolida_hoy": float(fila["util_consolida"]),
        }
    )

check_df = pd.DataFrame(check)
display(check_df)

base = check_df[check_df["motor"].isin(DEBE_REPRODUCIR)]
reproduce = bool(
    len(base)
    and (base["pallets_fase1"] == base["pallets_libre_hoy"]).all()
    and (base["util_fase1"] - base["util_libre_hoy"]).abs().max() < 1e-4
)
print("greedy y cinta (libre) reproducen la Fase 1:", reproduce)
if "aprendida#p1s1" in set(check_df["motor"]):
    print("aprendida#p1s1 es PPO de produccion; no se exige empate con la imitación de Fase 1")
if not reproduce:
    print("ATENCION: greedy/cinta no coinciden con Fase 1; no selles este resultado sin entender por que")

baseline tomado de: registro sellado (eval_fase1_multipallet)


,motor,pallets_fase1,pallets_libre_hoy,util_fase1,util_libre_hoy,pallets_consolida_hoy,util_consolida_hoy
0,greedy#p1s1,10,10,0.3725,0.3725,8,0.4986
1,aprendida#p1s1,10,10,0.3725,0.3725,8,0.4986
2,aprendida#p3s2,10,10,0.3725,0.3725,8,0.4986


greedy y cinta (libre) reproducen la Fase 1: True
aprendida#p1s1 es PPO de produccion; no se exige empate con la imitación de Fase 1


## Criterios de parada

El Paso A se considera **exitoso** si las cuatro condiciones se cumplen:

1. **Ahorra pallets.** El total de contenedores abiertos baja.
2. **No deja más carga fuera.** Ningún pedido empaca menos ítems que con el brazo libre.
3. **Todo sigue válido.** Cero violaciones geométricas o de peso.
4. **No empeora el cubo** en ningún pedido.

Y lo importante es lo que el resultado implica para los pasos siguientes:

- **Si es exitoso**, la brecha del multi-pallet estaba en el espacio de acciones y se cierra sin
  aprendizaje. El producto con dos pallets puede usar la envolvente. No se reabre el PPO.
- **Si no es exitoso**, la disciplina no basta. Eso queda como pista de producto; no se mezcla
  con más fine-tuning sobre el encoder v1 (ya cerrado).

La celda calcula además cuánto queda por ganar: compara el cubo consolidado con el techo de un solo
pallet que midió la Fase 0.

In [8]:
pallets_libre = int(deltas["pallets_libre"].sum())
pallets_cons = int(deltas["pallets_consolida"].sum())
ahorro = pallets_libre - pallets_cons

criterios = {
    "ahorra_pallets": bool(ahorro > 0),
    "no_deja_mas_items_fuera": bool((deltas["fuera_delta"] <= 0).all()),
    "todo_valido": bool(deltas["valido_consolida"].all() and deltas["valido_libre"].all()),
    "no_empeora_cubo": bool((deltas["util_delta"] >= -1e-9).all()),
}
paso_a_exitoso = all(criterios.values())

print("Criterios de parada")
for nombre, ok in criterios.items():
    print(f"  {'OK ' if ok else 'NO '} {nombre}")
print()
print(f"pallets abiertos: {pallets_libre} (libre) -> {pallets_cons} (consolida), ahorro {ahorro}")
print(f"cubo medio: {deltas['util_libre'].mean():.1%} -> {deltas['util_consolida'].mean():.1%}")
print()
print("PASO A EXITOSO" if paso_a_exitoso else "PASO A NO CONCLUYENTE")
print()

# Techo de referencia: el mejor cubo con un solo pallet que midio la Fase 0.
TECHO_FASE0_OFFLINE = 0.709   # offline best_fit_decreasing_3d, 1 pallet
TECHO_FASE0_ONLINE = 0.668    # online heuristico p1s1, 1 pallet
mejor_cons = float(deltas["util_consolida"].mean())
print(f"cubo consolidado (media)          : {mejor_cons:.1%}")
print(f"techo Fase 0 online 1 pallet      : {TECHO_FASE0_ONLINE:.1%}")
print(f"techo Fase 0 offline 1 pallet     : {TECHO_FASE0_OFFLINE:.1%}")
print(f"brecha restante vs offline 1 pallet: {max(0.0, TECHO_FASE0_OFFLINE - mejor_cons):.1%}")

Criterios de parada
  OK  ahorra_pallets
  OK  no_deja_mas_items_fuera
  OK  todo_valido
  OK  no_empeora_cubo

pallets abiertos: 30 (libre) -> 24 (consolida), ahorro 6
cubo medio: 37.3% -> 49.9%

PASO A EXITOSO

cubo consolidado (media)          : 49.9%
techo Fase 0 online 1 pallet      : 66.8%
techo Fase 0 offline 1 pallet     : 70.9%
brecha restante vs offline 1 pallet: 21.0%


## Sellado en el registro de versiones

Sellar convierte este resultado en una versión inmutable:

- copia el reporte a `artifacts/runs/eval_pasoA_consolidacion/vN/report.json`;
- guarda un `manifest.json` con el commit de git, la huella sha256 del notebook, las versiones de
  Python y torch, y las cifras de cabecera;
- añade la entrada al índice `artifacts/registry.json`.

Una versión sellada no se reescribe nunca. Si reejecutas el notebook sin cambiar nada, el registro
detecta que el código y las cifras son idénticos y **no** crea una versión nueva; si algo cambió,
sale `v2` y la anterior queda intacta.

El estado se marca `validado` solo si los cuatro criterios pasaron. Antes de sellar se vuelve a
comprobar que los checkpoints no se movieron: este flujo no reentrena nada.

In [9]:
for name, path in MODELS.items():
    assert path.stat().st_mtime == MTIMES_ANTES[name], f"{name} cambio: este flujo no reentrena"
print("checkpoints intactos:", *MODELS, sep="\n  ")

report = {
    "paso": "A",
    "pregunta": "cuanto del defecto multi-pallet se cierra sin reentrenar",
    "containers_available": CONTAINERS,
    "order_ids": ORDER_IDS,
    "encoder": {"feature_version": FEATURE_VERSION, "feature_dim": FEATURE_DIM},
    "checkpoints": {name: str(path) for name, path in MODELS.items()},
    "arms": [
        {
            "label": a.label,
            "family": a.family,
            "discipline": a.discipline,
            "lookahead_p": a.lookahead_p,
            "select_s": a.select_s,
            "model_path": a.model_path,
        }
        for a in PASO_A_ARMS
    ],
    "runs": rows,
    "pair_deltas": deltas.to_dict(orient="records"),
    "per_engine": resumen.to_dict(orient="records"),
    "baseline_fase1": {"origen": origen, "reproduce": reproduce, "check": check},
    "criterios": criterios,
    "paso_a_exitoso": paso_a_exitoso,
    "nota": "no reentrena, no toca encoder v1 ni checkpoints ni defaults del execute",
}

headline = {
    "pallets_libre": pallets_libre,
    "pallets_consolida": pallets_cons,
    "pallets_ahorrados": ahorro,
    "util_libre": round(float(deltas["util_libre"].mean()), 6),
    "util_consolida": round(float(deltas["util_consolida"].mean()), 6),
    "items_fuera_consolida": int(deltas["fuera_consolida"].sum()),
    "paso_a_exitoso": paso_a_exitoso,
}

entry = seal_run(
    FLOW_ID,
    title="Paso A: disciplina de consolidacion en multi-pallet (sin reentrenar)",
    source=NOTEBOOK,
    report=report,
    headline=headline,
    status=STATUS_VALIDADO if paso_a_exitoso else STATUS_EN_CURSO,
    notes=(
        "Envolvente ConsolidatingPolicy sobre greedy y RL de produccion. "
        "Diseno emparejado libre vs consolida. Sin reentrenamiento. PPO cerrado."
    ),
)
print("\nversion sellada:", entry["version"], "estado:", entry["status"])
print("carpeta:", entry["path"])
print()
print("Registro de flujos:")
display(pd.DataFrame(registry_table()))

checkpoints intactos:
  online_policy_ml/artifacts/models/mlp_v1_p1s1_ppo.pt
  online_policy_ml/artifacts/models/mlp_v1_p3s2.pt

version sellada: 3 estado: validado
carpeta: artifacts/runs/eval_pasoA_consolidacion/v3

Registro de flujos:


,flujo,titulo,versiones,ultima,estado,sellado,fuente
0,eval_fase0_baseline_p3s2,Fase 0 complemento: baseline greedy al mismo p...,1,v1,validado,2026-09-11,scripts/run_holdout_offline_vs_online.py
1,eval_fase0_holdout,Fase 0: comparacion de producto offline vs onl...,1,v1,validado,2026-09-11,scripts/run_holdout_offline_vs_online.py
2,eval_fase1_multipallet,Fase 1: multi-pallet con dos contenedores iden...,1,v1,validado,2026-09-11,scripts/run_holdout_offline_vs_online.py
3,eval_pasoA_consolidacion,Paso A: disciplina de consolidacion en multi-p...,3,v3,validado,2026-09-15,online_policy_ml/notebooks/07_multipallet_firs...
4,eval_pasoC_producto,Paso C: execute de producto vs Paso A consolidado,2,v2,validado,2026-09-13,online_policy_ml/notebooks/11_producto_multipa...
5,ml_fase1_datos_splits,Fase 1: datos BED-BPP y splits,1,v1,validado,2026-09-11,online_policy_ml/notebooks/01_datos_y_splits.i...
6,ml_fase2_etiquetas_p2o,Fase 2: transiciones P2O etiquetadas por el ma...,1,v1,validado,2026-09-11,online_policy_ml/notebooks/02_etiquetas_p2o.ipynb
7,ml_fase3_entrenar_exportar,"Fase 3: entrenar, validar y exportar checkpoints",1,v1,validado,2026-09-11,online_policy_ml/notebooks/03_entrenar_validar...
8,ml_fase4_escalar,Fase 4: escalar a pedidos medianos y revalidar,1,v1,validado,2026-09-11,online_policy_ml/notebooks/04_escalar_revalida...
9,ml_fase5_receding_p3s2,Fase 5: maestro receding-horizon para el regim...,1,v1,validado,2026-09-11,online_policy_ml/notebooks/05_receding_horizon...
